In [1]:
from openff.evaluator.datasets.datasets import PhysicalPropertyDataSet
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import Draw
IPythonConsole.ipython_useSVG=True
import tqdm
from collections import defaultdict

In [2]:
def sanitize_smiles(smi):
    return Chem.MolToSmiles(Chem.MolFromSmiles(smi))

In [4]:
SIDE_CHAIN_ANALOGS = {
    "ala": "C",
    "val": "CCC",
    "leu": "CC(C)C",
    "ile": "CCCC",
    "met": "CCSC",
    "phe": "Cc1ccccc1",
    "trp": "CC1=CNC2=CC=CC=C12",
    "hid": "CC1=CN=CN1",
    "hie": "CC1N=CNC=1",
    "lys": "CCCCN",
    "arg": "CCCNC(=N)N",
    "asp": "CC(=O)O",
    "glu": "CCC(=O)O",
    "ser": "CO",
    "thr": "CCO",
    "cys": "CS",
    "tyr": "Cc1ccc(O)cc1",
    "asn": "CC(=O)N",
    "gln": "CCC(=O)N",
}
SIDE_CHAIN_ANALOGS_SAN = {
    sanitize_smiles(v): k
    for k, v in SIDE_CHAIN_ANALOGS.items()
}
matches = defaultdict(list)

In [5]:
df = pd.read_csv("../output/initial-filtered.csv", index_col=0)

In [6]:
for _, row in tqdm.tqdm(df.iterrows()):
    smiles = [row["Component 1"]]
    if row["N Components"] == 2:
        smiles.append(row["Component 2"])
    sanitized_smiles = [sanitize_smiles(x) for x in smiles]
    for sanitized in sanitized_smiles:
        if sanitized in SIDE_CHAIN_ANALOGS_SAN:
            matches[SIDE_CHAIN_ANALOGS_SAN[sanitized]].append(row)

9218it [00:00, 17012.65it/s]


In [7]:
matching_rows = []
for key, rows in matches.items():
    for row in rows:
        row["match"] = key
        matching_rows.append(row)

matching_df = pd.DataFrame(matching_rows)

In [8]:
matching_df.columns

Index(['Id', 'Temperature (K)', 'Pressure (kPa)', 'Phase', 'N Components',
       'Component 1', 'Role 1', 'Mole Fraction 1', 'Exact Amount 1',
       'Component 2', 'Role 2', 'Mole Fraction 2', 'Exact Amount 2',
       'Component 3', 'Role 3', 'Mole Fraction 3', 'Exact Amount 3',
       'EnthalpyOfMixing Value (kJ / mol)',
       'EnthalpyOfMixing Uncertainty (kJ / mol)',
       'DielectricConstant Value ()', 'DielectricConstant Uncertainty ()',
       'Density Value (g / ml)', 'Density Uncertainty (g / ml)',
       'ExcessMolarVolume Value (cm ** 3 / mol)',
       'ExcessMolarVolume Uncertainty (cm ** 3 / mol)', 'Source', 'match'],
      dtype='object')

In [9]:
len(matching_df)

1147

In [10]:
matching_df["EnthalpyOfMixing Value (kJ / mol)"].notna().sum()

293

In [11]:
matching_df["Density Value (g / ml)"].notna().sum()

854

In [12]:
matching_df.groupby("match").count()

,Id,Temperature (K),Pressure (kPa),Phase,N Components,Component 1,Role 1,Mole Fraction 1,Exact Amount 1,Component 2,...,Exact Amount 3,EnthalpyOfMixing Value (kJ / mol),EnthalpyOfMixing Uncertainty (kJ / mol),DielectricConstant Value (),DielectricConstant Uncertainty (),Density Value (g / ml),Density Uncertainty (g / ml),ExcessMolarVolume Value (cm ** 3 / mol),ExcessMolarVolume Uncertainty (cm ** 3 / mol),Source
match,,,,,,,,,,,,,,,,,,,,,
asp,71,71,71,71,71,71,71,71,0,70,...,0,15,15,0,0,56,56,0,0,71
glu,28,28,28,28,28,28,28,28,0,27,...,0,0,0,0,0,28,28,0,0,28
lys,37,37,37,37,37,37,37,37,0,36,...,0,6,6,0,0,31,31,0,0,37
phe,277,277,277,277,277,277,277,277,0,276,...,0,75,75,0,0,202,202,0,0,277
ser,256,256,256,256,256,256,256,256,0,255,...,0,80,80,0,0,176,176,0,0,256
thr,451,451,451,451,451,451,451,451,0,450,...,0,117,117,0,0,334,334,0,0,451
tyr,27,27,27,27,27,27,27,27,0,26,...,0,0,0,0,0,27,27,0,0,27


In [13]:
matching_df.to_csv("aa-analogs.csv")